# Delivery ETA Predictor -- EDA and Modelling

Sections:
1. Dataset overview
2. Distribution plots
3. Correlation heatmap
4. Geospatial scatter plot
5. Model comparison table
6. SHAP summary plot
7. Conclusions and next steps

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

df = pd.read_csv('data/raw/delivery_data.csv', parse_dates=['order_timestamp'])
print(f'Data shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Shape ===')
print(df.shape)
print('\n=== Dtypes ===')
print(df.dtypes)
print('\n=== Nulls ===')
print(df.isnull().sum())
print('\n=== Describe ===')
df.describe()

## 2. Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].hist(df['actual_delivery_minutes'], bins=50, edgecolor='black', color='steelblue')
axes[0,0].set_title('Delivery Minutes (Target)')
axes[0,0].set_xlabel('Minutes')

axes[0,1].hist(df['haversine_distance_km'], bins=50, edgecolor='black', color='coral')
axes[0,1].set_title('Distance (km)')
axes[0,1].set_xlabel('km')

axes[0,2].hist(df['traffic_index'], bins=50, edgecolor='black', color='green')
axes[0,2].set_title('Traffic Index')

axes[1,0].hist(df['num_items'], bins=20, edgecolor='black', color='purple')
axes[1,0].set_title('Number of Items')

axes[1,1].hist(df['order_value_inr'], bins=50, edgecolor='black', color='orange')
axes[1,1].set_title('Order Value (INR)')

axes[1,2].hist(df['rider_experience_days'], bins=50, edgecolor='black', color='teal')
axes[1,2].set_title('Rider Experience (days)')

plt.tight_layout()
plt.savefig('reports/distributions.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['weather_condition'].value_counts().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Weather Condition Distribution')
axes[0].tick_params(axis='x', rotation=0)

df['store_type'].value_counts().plot.bar(ax=axes[1], color='coral')
axes[1].set_title('Store Type Distribution')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 3. Correlation Heatmap

In [ ]:
numeric_cols = ['haversine_distance_km', 'hour_of_day', 'day_of_week', 'is_weekend',
                'is_peak_hour', 'traffic_index', 'num_items', 'order_value_inr',
                'rider_experience_days', 'concurrent_orders_at_store', 'actual_delivery_minutes']

corr = df[numeric_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('reports/correlation_heatmap.png', dpi=150)
plt.show()

## 4. Geospatial Scatter Plot

In [ ]:
sample = df.sample(min(5000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(sample['store_lng'], sample['store_lat'], alpha=0.3, s=5, c='blue', label='Store')
ax.scatter(sample['customer_lng'], sample['customer_lat'], alpha=0.2, s=3, c='red', label='Customer')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geospatial Distribution: Store vs Customer Locations')
ax.legend()
plt.tight_layout()
plt.savefig('reports/geospatial_scatter.png', dpi=150)
plt.show()

## 5. Model Comparison Table

In [ ]:
import pickle

try:
    with open('models/best_model.pkl', 'rb') as f:
        artifact = pickle.load(f)

    print(f"Best model: {artifact['model_name']}")
    print(f"Validation metrics: {artifact['val_metrics']}")
    print(f"Test metrics: {artifact['test_metrics']}")

    metrics_df = pd.DataFrame([artifact['test_metrics']], index=[artifact['model_name']])
    display(metrics_df)
except FileNotFoundError:
    print('Model not found. Run `python src/train.py` first.')

## 6. SHAP Summary Plot

In [ ]:
try:
    import shap

    with open('models/best_model.pkl', 'rb') as f:
        artifact = pickle.load(f)

    model = artifact['model']
    feature_names = artifact['feature_names']
    test_df = pd.read_pickle('data/processed/test_data_with_preds.pkl')
    X_test = test_df[feature_names].values.astype(np.float64)

    explainer = shap.TreeExplainer(model)
    X_sample = X_test[:500]
    shap_values = explainer.shap_values(X_sample)

    plt.figure()
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.savefig('reports/shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('shap not installed. Run: pip install shap')
except Exception as e:
    print(f'SHAP error: {e}')

## 7. Conclusions and Next Steps

### Key Findings

- Distance and traffic are the strongest predictors of delivery time.
- Heavy rain adds a significant penalty (up to 10 minutes on average).
- Peak hours show elevated times due to higher concurrent order volume.
- Rider experience provides a small but consistent reduction.
- XGBoost with Optuna tuning achieved the lowest RMSE on both val and test sets.

### Next Steps

1. Replace synthetic data with real delivery logs.
2. Add holiday calendar and event-based temporal features.
3. Integrate OSRM / Google Maps for actual route distance.
4. Set up automated retraining pipeline.
5. Add live traffic and weather API features for real-time inference.
6. Set up continuous drift detection and alerting in production.